# 07 - Train LSTM Future Risk Model

This notebook trains an LSTM model using patient-level exam sequences.

Each patient sequence contains exam-level CNN features, asymmetry features, view availability masks, and recency weights. The model predicts 1-year, 2-year, 3-year, 4-year, and 5-year future risk.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Imports

import os
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [3]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

In [4]:
# File paths

SEQUENCE_PATH = "/content/drive/MyDrive/EMBED/Features/embed_lstm_patient_sequences.npy"
LABEL_PATH = "/content/drive/MyDrive/EMBED/Features/embed_lstm_patient_labels.npy"
MASK_PATH = "/content/drive/MyDrive/EMBED/Features/embed_lstm_sequence_masks.npy"
METADATA_PATH = "/content/drive/MyDrive/EMBED/Features/embed_lstm_patient_metadata.csv"

MODEL_SAVE_PATH = "/content/drive/MyDrive/EMBED/Models/best_lstm_future_risk.pth"

In [5]:
# Load LSTM-ready data

X = np.load(SEQUENCE_PATH)
y = np.load(LABEL_PATH)
sequence_masks = np.load(MASK_PATH)
metadata = pd.read_csv(METADATA_PATH)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Mask shape:", sequence_masks.shape)
print("Metadata shape:", metadata.shape)

metadata.head()

X shape: (40, 13, 6149)
y shape: (40, 5)
Mask shape: (40, 13)
Metadata shape: (40, 5)


,empi_anon,num_exams,first_exam_date,last_exam_date,max_sequence_length
0,11057159,7,2013-03-05,2020-03-02,13
1,11354401,6,2013-12-14,2020-05-21,13
2,12249159,13,2013-02-22,2020-04-20,13
3,18069268,2,2018-03-17,2020-05-07,13
4,18954563,3,2018-04-05,2019-03-04,13


In [6]:
print("1yr positives:", y[:,0].sum())
print("2yr positives:", y[:,1].sum())
print("3yr positives:", y[:,2].sum())
print("4yr positives:", y[:,3].sum())
print("5yr positives:", y[:,4].sum())

1yr positives: 3.0
2yr positives: 9.0
3yr positives: 14.0
4yr positives: 16.0
5yr positives: 18.0


In [7]:
from sklearn.model_selection import KFold

kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

folds = list(kfold.split(X))

print("Number of folds:", len(folds))

for fold, (train_idx, test_idx) in enumerate(folds, start=1):
    print(f"Fold {fold}")
    print("Train patients:", len(train_idx))
    print("Test patients:", len(test_idx))
    print("Test indices:", test_idx)
    print("-" * 40)

Number of folds: 5
Fold 1
Train patients: 32
Test patients: 8
Test indices: [ 4 12 15 16 19 26 27 37]
----------------------------------------
Fold 2
Train patients: 32
Test patients: 8
Test indices: [ 6  8  9 13 25 31 34 39]
----------------------------------------
Fold 3
Train patients: 32
Test patients: 8
Test indices: [ 0  1  5 11 17 24 29 33]
----------------------------------------
Fold 4
Train patients: 32
Test patients: 8
Test indices: [ 2  3 21 23 30 32 35 36]
----------------------------------------
Fold 5
Train patients: 32
Test patients: 8
Test indices: [ 7 10 14 18 20 22 28 38]
----------------------------------------


In [8]:
class FutureRiskDataset(Dataset):

    def __init__(self, X, y, masks):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.masks = torch.tensor(masks, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.masks[idx]

In [9]:
class FutureRiskLSTM(nn.Module):

    def __init__(
        self,
        input_size=6149,
        compressed_size=256,
        hidden_size=128,
        num_layers=1,
        output_size=5,
        dropout=0.3
    ):
        super(FutureRiskLSTM, self).__init__()

        self.feature_compressor = nn.Sequential(
            nn.Linear(input_size, compressed_size),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.lstm = nn.LSTM(
            input_size=compressed_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.output_head = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, output_size)
        )

    def forward(self, x, mask):
        x = self.feature_compressor(x)

        lstm_output, _ = self.lstm(x)

        lengths = mask.sum(dim=1).long()
        last_indices = lengths - 1

        batch_indices = torch.arange(
            x.size(0),
            device=x.device
        )

        last_outputs = lstm_output[
            batch_indices,
            last_indices
        ]

        logits = self.output_head(last_outputs)

        return logits

In [10]:
model = FutureRiskLSTM().to(device)

sample_X = torch.tensor(X[:4], dtype=torch.float32).to(device)
sample_mask = torch.tensor(sequence_masks[:4], dtype=torch.float32).to(device)

with torch.no_grad():
    sample_logits = model(sample_X, sample_mask)

print(sample_logits.shape)

torch.Size([4, 5])


In [11]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for X_batch, y_batch, mask_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        mask_batch = mask_batch.to(device)

        optimizer.zero_grad()

        logits = model(X_batch, mask_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    all_probs = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch, mask_batch in dataloader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            mask_batch = mask_batch.to(device)

            logits = model(X_batch, mask_batch)
            loss = criterion(logits, y_batch)

            probs = torch.sigmoid(logits)

            total_loss += loss.item()
            all_probs.append(probs.cpu().numpy())
            all_labels.append(y_batch.cpu().numpy())

    all_probs = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)

    return total_loss / len(dataloader), all_probs, all_labels

In [12]:
num_epochs = 30
fold_results = []

BEST_MODEL_PATH = "/content/drive/MyDrive/EMBED/models/best_lstm_future_risk.pth"

best_overall_loss = float("inf")

for fold, (train_idx, test_idx) in enumerate(folds, start=1):
    print(f"\n===== Fold {fold} =====")

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    mask_train, mask_test = sequence_masks[train_idx], sequence_masks[test_idx]

    train_dataset = FutureRiskDataset(X_train, y_train, mask_train)
    test_dataset = FutureRiskDataset(X_test, y_test, mask_test)

    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

    model = FutureRiskLSTM().to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-4,
        weight_decay=1e-4
    )

    best_test_loss = float("inf")
    best_probs = None
    best_labels = None

    for epoch in range(num_epochs):
        train_loss = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device
        )

        test_loss, probs, labels = evaluate(
            model,
            test_loader,
            criterion,
            device
        )

        if test_loss < best_test_loss:

          best_test_loss = test_loss

          best_probs = probs
          best_labels = labels

          if test_loss < best_overall_loss:

              best_overall_loss = test_loss

              torch.save(
                  model.state_dict(),
                  BEST_MODEL_PATH
              )

        print(
            f"Saved new best model "
            f"(loss={test_loss:.4f})"
        )

        print(
            f"Epoch {epoch+1:02d}/{num_epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Test Loss: {test_loss:.4f}"
        )

    fold_results.append({
        "fold": fold,
        "best_test_loss": best_test_loss,
        "probs": best_probs,
        "labels": best_labels
    })

print("\nTraining complete.")


===== Fold 1 =====
Saved new best model (loss=0.6791)
Epoch 01/30 | Train Loss: 0.6863 | Test Loss: 0.6791
Saved new best model (loss=0.6650)
Epoch 02/30 | Train Loss: 0.6785 | Test Loss: 0.6650
Saved new best model (loss=0.6529)
Epoch 03/30 | Train Loss: 0.6714 | Test Loss: 0.6529
Saved new best model (loss=0.6394)
Epoch 04/30 | Train Loss: 0.6625 | Test Loss: 0.6394
Saved new best model (loss=0.6224)
Epoch 05/30 | Train Loss: 0.6572 | Test Loss: 0.6224
Saved new best model (loss=0.6052)
Epoch 06/30 | Train Loss: 0.6509 | Test Loss: 0.6052
Saved new best model (loss=0.5932)
Epoch 07/30 | Train Loss: 0.6357 | Test Loss: 0.5932
Saved new best model (loss=0.5789)
Epoch 08/30 | Train Loss: 0.6335 | Test Loss: 0.5789
Saved new best model (loss=0.5603)
Epoch 09/30 | Train Loss: 0.6198 | Test Loss: 0.5603
Saved new best model (loss=0.5412)
Epoch 10/30 | Train Loss: 0.6082 | Test Loss: 0.5412
Saved new best model (loss=0.5288)
Epoch 11/30 | Train Loss: 0.5971 | Test Loss: 0.5288
Saved new be

In [13]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

risk_names = ["1yr", "2yr", "3yr", "4yr", "5yr"]

metrics_rows = []

for result in fold_results:
    fold = result["fold"]
    probs = result["probs"]
    labels = result["labels"]

    preds = (probs >= 0.5).astype(int)

    for i, risk_name in enumerate(risk_names):
        y_true = labels[:, i]
        y_pred = preds[:, i]
        y_prob = probs[:, i]

        try:
            auc = roc_auc_score(y_true, y_prob)
        except ValueError:
            auc = np.nan

        metrics_rows.append({
            "fold": fold,
            "risk_window": risk_name,
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "roc_auc": auc
        })

metrics_df = pd.DataFrame(metrics_rows)

metrics_df

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist

,fold,risk_window,accuracy,precision,recall,f1,roc_auc
0,1,1yr,1.000,0.0,0.000000,0.000000,NaN
1,1,2yr,1.000,0.0,0.000000,0.000000,NaN
2,1,3yr,0.875,0.0,0.000000,0.000000,NaN
3,1,4yr,0.625,0.0,0.000000,0.000000,0.142857
4,1,5yr,0.750,1.0,0.333333,0.500000,0.733333
5,2,1yr,0.750,0.0,0.000000,0.000000,0.750000
6,2,2yr,0.375,0.0,0.000000,0.000000,0.866667
7,2,3yr,0.375,0.0,0.000000,0.000000,0.200000
8,2,4yr,0.375,0.0,0.000000,0.000000,0.133333
9,2,5yr,0.375,0.0,0.000000,0.000000,0.266667


In [14]:
summary_metrics = (
    metrics_df
    .groupby("risk_window")
    [["accuracy", "precision", "recall", "f1", "roc_auc"]]
    .mean()
    .reset_index()
)

summary_metrics

,risk_window,accuracy,precision,recall,f1,roc_auc
0,1yr,0.925,0.0,0.000000,0.000000,0.875000
1,2yr,0.775,0.0,0.000000,0.000000,0.888889
2,3yr,0.625,0.0,0.000000,0.000000,0.697619
3,4yr,0.600,0.2,0.133333,0.160000,0.615238
4,5yr,0.725,0.8,0.446667,0.547619,0.760000
